# DBSCAN silhouette scores: score, grouped-only and coverage

DBSCAN labels many feature names as noise, so one silhouette number is not enough. This notebook sweeps `eps` and `min_samples` and reports, for every setting:

| Column | Meaning |
|---|---|
| **Clustered / Coverage** | feature names in a cluster, and their share of all feature names |
| **Score** | silhouette over all $n$ feature names, noise counted as one-item clusters (so noise contributes 0). Used to rank settings |
| **$S_g$ (grouped only)** | silhouette over the clustered feature names only, noise removed first. Measures how tight the clusters are |
| **Bound** | $\text{coverage}\times S_g$, an upper limit for the score |
| **$\bar S_g'$** | mean silhouette of the clustered feature names while the noise feature names are still present as one-item clusters |
| **$\delta$** | $S_g - \bar S_g'$, the extra drop caused by noise feature names acting as nearby clusters |

The conversion is $\text{score} = \text{coverage}\cdot\bar S_g' = \text{coverage}\cdot(S_g-\delta)$, which this notebook checks numerically for every row.

Reference bands for an average silhouette (Kaufman and Rousseeuw, rule of thumb): above 0.70 strong, 0.51 to 0.70 reasonable, 0.26 to 0.50 weak, 0.25 or below no substantial structure.

In [1]:
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML, display
from plotly.subplots import make_subplots
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_samples

MATRIX_PATH = Path("data/similarity_expanded.npz")  # use data/similarity.npz for the original 445 feature names
EPS_VALUES = np.round(np.arange(0.20, 0.62, 0.02), 2)
MIN_SAMPLES_VALUES = [2, 3, 4, 5, 6, 8]
PLOT_MIN_SAMPLES = 2  # min_samples used for the eps curves below (the best value in every row of the table)
MIN_COVERAGE = 0.30  # floor used when choosing a setting; set to 0 to rank purely by score
TOP_N = 10

if not MATRIX_PATH.is_file():
    MATRIX_PATH = Path("CTFDS-Project") / MATRIX_PATH
with np.load(MATRIX_PATH) as data:
    similarity = data["similarity"]
    feature_names = data["titles"].tolist()

n = len(feature_names)
assert similarity.shape == (n, n)
assert np.allclose(similarity, similarity.T), "similarity must be symmetric"
distance = np.clip(1 - similarity, 0, 1)
np.fill_diagonal(distance, 0)
print(f"Loaded {n} feature names from {MATRIX_PATH}")

Loaded 664 feature names from data/similarity_expanded.npz


## 1. Run the sweep

Each noise item is relabelled as its own one-item cluster before the silhouette is computed (the same rule as `6_dbscan.ipynb`). `silhouette_samples` gives one value per item, so the score and both grouped-only variants come from the same run.

In [2]:
def noise_as_singletons(labels):
    relabelled = labels.copy()
    noise = np.where(relabelled == -1)[0]
    relabelled[noise] = np.arange(relabelled.max() + 1, relabelled.max() + 1 + len(noise))
    return relabelled


def evaluate(labels):
    clustered = labels != -1
    n_clusters = len(set(labels[clustered].tolist()))
    if n_clusters < 2 or n_clusters >= clustered.sum():
        return None  # silhouette undefined

    per_item = silhouette_samples(distance, noise_as_singletons(labels), metric="precomputed")
    score = float(per_item.mean())
    s_g_prime = float(per_item[clustered].mean())  # clustered feature names, noise still present

    s_g = float(
        silhouette_samples(distance[np.ix_(clustered, clustered)], labels[clustered], metric="precomputed").mean()
    )
    coverage = float(clustered.mean())
    assert np.isclose(score, coverage * s_g_prime), "score must equal coverage x S_g'"
    return {
        "n_clusters": n_clusters,
        "clustered": int(clustered.sum()),
        "coverage": coverage,
        "largest": int(np.bincount(labels[clustered]).max()),
        "score": score,
        "s_g": s_g,
        "bound": coverage * s_g,
        "s_g_prime": s_g_prime,
        "delta": s_g - s_g_prime,
    }


results = []
for eps in EPS_VALUES:
    for min_samples in MIN_SAMPLES_VALUES:
        labels = DBSCAN(eps=float(eps), min_samples=min_samples, metric="precomputed").fit_predict(distance)
        metrics = evaluate(labels)
        if metrics is not None:
            results.append({"eps": float(eps), "min_samples": min_samples, **metrics})

print(f"{len(results)} of {len(EPS_VALUES) * len(MIN_SAMPLES_VALUES)} settings give at least 2 clusters")

94 of 126 settings give at least 2 clusters


## 2. Best `min_samples` for each `eps`

For each `eps`, the row with the highest score. The table shows how the score, the grouped-only silhouette and the coverage move against each other as `eps` grows.

In [3]:
def show(figure):
    try:
        figure.show(renderer="notebook_connected")
    except ValueError:
        display(HTML(figure.to_html(full_html=False, include_plotlyjs=True)))


best_per_eps = []
for eps in sorted({row["eps"] for row in results}):
    best_per_eps.append(max((r for r in results if r["eps"] == eps), key=lambda r: r["score"]))

columns = [
    ("eps", "eps", ".2f"), ("min_samples", "min_samples", "d"), ("Clusters", "n_clusters", "d"),
    ("Clustered", "clustered", "d"), ("Coverage %", "coverage", ".1f"), ("Largest", "largest", "d"),
    ("Score", "score", ".4f"), ("S_g (grouped only)", "s_g", ".4f"), ("Bound = cov x S_g", "bound", ".4f"),
    ("S_g' (noise present)", "s_g_prime", ".4f"), ("delta", "delta", ".4f"),
]


def cell(row, key, spec):
    value = 100 * row[key] if key == "coverage" else row[key]
    return format(value, spec)


def make_table(rows, title, highlight=None):
    fill = ["#fde8e7" if highlight is not None and r is highlight else "white" for r in rows]
    figure = go.Figure(go.Table(
        header=dict(values=[c[0] for c in columns], fill_color="#2b2b2b", font=dict(color="white", size=11), align="center"),
        cells=dict(values=[[cell(r, key, spec) for r in rows] for _, key, spec in columns],
                   fill_color=[fill] * len(columns), align="center", height=26),
    ))
    figure.update_layout(title=dict(text=title, x=0), width=1100, height=130 + 27 * len(rows), margin=dict(t=60, b=10))
    show(figure)


make_table(best_per_eps, "Best min_samples per eps (ranked by score)")

This graph shows how the distance threshold $\varepsilon$ changes the noise, the group sizes, the coverage and the score. It uses one fixed `min_samples` (`PLOT_MIN_SAMPLES`) so that every point of a curve comes from the same setting; the table above picks the best `min_samples` per `eps`, which can jump between values.

- **Top:** score, $S_g$ and coverage, all between 0 and 1 on one axis. The dotted lines are the silhouette bands; each label sits *inside* its band, below its line.
- **Bottom:** number of noise feature names, number of clusters and size of the largest cluster.

In [4]:
curve = sorted((r for r in results if r["min_samples"] == PLOT_MIN_SAMPLES), key=lambda r: r["eps"])
eps_axis = [r["eps"] for r in curve]

figure = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1, row_heights=[0.58, 0.42],
    subplot_titles=("Silhouette and coverage", "Noise, clusters and largest cluster"),
)
top = [
    ("Score (noise as singletons)", [r["score"] for r in curve], "#cc1f1a", "solid"),
    ("S_g (grouped only)", [r["s_g"] for r in curve], "#005f73", "solid"),
    ("Coverage", [r["coverage"] for r in curve], "#6c757d", "dash"),
]
for name, values, color, dash in top:
    figure.add_trace(go.Scatter(x=eps_axis, y=values, name=name, mode="lines+markers",
                                line=dict(color=color, width=3, dash=dash)), row=1, col=1)
for edge, label in [(0.25, "no substantial structure (up to 0.25)"), (0.50, "weak (0.26 to 0.50)"), (0.70, "reasonable (0.51 to 0.70)")]:
    figure.add_hline(y=edge, line=dict(color="#c9c8c0", width=1, dash="dot"), annotation_text=label,
                     annotation_position="bottom left", annotation_font_size=10, row=1, col=1)
figure.add_hline(y=0.70, line=dict(width=0), annotation_text="strong (above 0.70)",
                 annotation_position="top left", annotation_font_size=10, row=1, col=1)

bottom = [
    ("Noise", [n - r["clustered"] for r in curve], "#9b2226", "solid"),
    ("Clusters", [r["n_clusters"] for r in curve], "#0a9396", "solid"),
    ("Largest cluster", [r["largest"] for r in curve], "#ee9b00", "solid"),
]
for name, values, color, dash in bottom:
    figure.add_trace(go.Scatter(x=eps_axis, y=values, name=name, mode="lines+markers",
                                line=dict(color=color, width=3, dash=dash)), row=2, col=1)

figure.update_layout(
    title=dict(text=f"DBSCAN as eps grows (min_samples = {PLOT_MIN_SAMPLES})", x=0),
    template="simple_white", width=1000, height=800, legend=dict(orientation="h", y=-0.12),
)
figure.update_yaxes(title_text="silhouette / coverage", range=[-0.15, 1.0], row=1, col=1)
figure.update_yaxes(title_text="feature names", row=2, col=1)
figure.update_xaxes(title_text="eps", row=2, col=1)
show(figure)

## 3. Top settings under the coverage floor

Rule: drop settings below `MIN_COVERAGE`, rank the rest by score (ties: more coverage, then fewer clusters). The best row is highlighted.

In [5]:
eligible = [r for r in results if r["coverage"] >= MIN_COVERAGE]
if not eligible:
    raise ValueError(f"No setting reaches coverage >= {MIN_COVERAGE:.0%}; lower MIN_COVERAGE or raise eps.")
ranked = sorted(eligible, key=lambda r: (-r["score"], -r["coverage"], r["n_clusters"]))
best = ranked[0]

print(f"{len(eligible)} of {len(results)} settings reach coverage >= {MIN_COVERAGE:.0%}")
make_table(ranked[:TOP_N], f"Top {min(TOP_N, len(ranked))} settings with coverage >= {MIN_COVERAGE:.0%}", highlight=best)
print(
    f"Selected: eps={best['eps']:g}, min_samples={best['min_samples']} | "
    f"{best['clustered']} of {n} feature names clustered ({100 * best['coverage']:.1f}%) in {best['n_clusters']} clusters | "
    f"score={best['score']:.4f}, S_g={best['s_g']:.4f}"
)

print("\nBest setting for several coverage floors:")
for floor in (0.0, 0.3, 0.4, 0.5, 0.6):
    pool = [r for r in results if r["coverage"] >= floor]
    if pool:
        r = max(pool, key=lambda r: r["score"])
        print(f"  coverage >= {floor:.0%}: eps={r['eps']:g}, min_samples={r['min_samples']}, "
              f"clustered={r['clustered']} ({100 * r['coverage']:.1f}%), score={r['score']:.4f}, S_g={r['s_g']:.4f}")

40 of 94 settings reach coverage >= 30%


Selected: eps=0.4, min_samples=2 | 248 of 664 feature names clustered (37.3%) in 82 clusters | score=0.1258, S_g=0.4479

Best setting for several coverage floors:
  coverage >= 0%: eps=0.4, min_samples=2, clustered=248 (37.3%), score=0.1258, S_g=0.4479
  coverage >= 30%: eps=0.4, min_samples=2, clustered=248 (37.3%), score=0.1258, S_g=0.4479
  coverage >= 40%: eps=0.42, min_samples=2, clustered=270 (40.7%), score=0.1100, S_g=0.3863
  coverage >= 50%: eps=0.48, min_samples=2, clustered=344 (51.8%), score=0.0779, S_g=0.2857
  coverage >= 60%: eps=0.52, min_samples=2, clustered=427 (64.3%), score=0.0533, S_g=0.1914


## 4. Conversion check for the selected setting

The identity $\text{score} = \text{coverage}\cdot\bar S_g'$ holds exactly (it is asserted for every row above). The grouped-only value $S_g$ is larger because it removes the noise feature names, so $\delta = S_g - \bar S_g'$ shows how much the nearby noise feature names cost the clustered ones.

In [6]:
print(f"coverage             = {best['coverage']:.4f}  ({best['clustered']} / {n})")
print(f"S_g (grouped only)   = {best['s_g']:.4f}")
print(f"upper bound          = coverage x S_g = {best['bound']:.4f}")
print(f"S_g' (noise present) = {best['s_g_prime']:.4f}   (delta = {best['delta']:.4f})")
print(f"score                = coverage x S_g' = {best['coverage'] * best['s_g_prime']:.4f}  (computed directly: {best['score']:.4f})")

coverage             = 0.3735  (248 / 664)
S_g (grouped only)   = 0.4479
upper bound          = coverage x S_g = 0.1673
S_g' (noise present) = 0.3367   (delta = 0.1111)
score                = coverage x S_g' = 0.1258  (computed directly: 0.1258)


## 5. Silhouette scoring: reading the values

Maps each silhouette value to the reference bands from the introduction. Read $S_g$ against the bands to judge the clusters; the score includes the noise zeros, so it is best read as a ranking, not against the bands (its band is shown for completeness).

In [7]:
BANDS = [
    (0.70, "strong"),
    (0.50, "reasonable"),
    (0.25, "weak"),
    (0.00, "no substantial structure"),
]


def band(value):
    if value < 0:
        return "negative (closer to another cluster)"
    return next(label for lower, label in BANDS if value > lower) if value > 0.25 else "no substantial structure"


print("Selected setting:")
print(f"  score = {best['score']:.4f}  -> {band(best['score'])}   (includes noise zeros; use for ranking)")
print(f"  S_g   = {best['s_g']:.4f}  -> {band(best['s_g'])}   (clusters only)")

scoring_rows = best_per_eps
header = f"{'eps':>5} {'cov %':>6} {'score':>8} {'S_g':>8}  S_g reading"
print("\nBest min_samples per eps:")
print(header)
for r in scoring_rows:
    print(f"{r['eps']:5.2f} {100 * r['coverage']:6.1f} {r['score']:8.4f} {r['s_g']:8.4f}  {band(r['s_g'])}")


Selected setting:
  score = 0.1258  -> no substantial structure   (includes noise zeros; use for ranking)
  S_g   = 0.4479  -> weak   (clusters only)

Best min_samples per eps:
  eps  cov %    score      S_g  S_g reading
 0.20    9.2   0.0609   0.7795  strong
 0.22   11.3   0.0718   0.7608  strong
 0.24   12.3   0.0772   0.7436  strong
 0.26   14.9   0.0866   0.7042  strong
 0.28   17.0   0.0915   0.6719  reasonable
 0.30   19.0   0.0973   0.6332  reasonable
 0.32   22.1   0.0982   0.5775  reasonable
 0.34   25.3   0.1013   0.5241  reasonable
 0.36   29.4   0.1071   0.5010  reasonable
 0.38   33.4   0.1184   0.4690  weak
 0.40   37.3   0.1258   0.4479  weak
 0.42   40.7   0.1100   0.3863  weak
 0.44   44.0   0.1070   0.3623  weak
 0.46   48.8   0.1031   0.3326  weak
 0.48   51.8   0.0779   0.2857  weak
 0.50   58.6   0.0689   0.2416  no substantial structure
 0.52   64.3   0.0533   0.1914  no substantial structure
 0.54   70.2   0.0216   0.1235  no substantial structure
 0.56   75.8  -